In [1]:
!apt-get update
!apt-get install openjdk-8-jdk-headless -qq
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease [24.6 kB]
Get:7 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,786 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,669 kB]
Get:12 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [12

In [3]:
from pyspark.sql.types import *

dataframe = spark.read.csv("challenge.csv" , header=True)
dataframe.show()

+---------------+--------------+-----------------+----------+
|     ip_address|       Country|      Domain Name|Bytes_used|
+---------------+--------------+-----------------+----------+
|  52.81.192.172|         China| odnoklassniki.ru|       463|
| 119.239.207.13|         China|         youtu.be|        51|
|  68.69.217.210|         China|        adobe.com|        10|
|   7.191.21.223|      Bulgaria|     linkedin.com|       853|
|   211.13.10.68|     Indonesia|          hud.gov|        29|
|   239.80.21.97|      Suriname|       smh.com.au|       218|
|106.214.106.233|       Jamaica|    amazonaws.com|        95|
| 127.242.24.138|         China| surveymonkey.com|       123|
|     99.2.6.139|Czech Republic|     geocities.jp|       322|
|   237.54.11.63|         China|       amazon.com|        83|
| 252.141.157.25|         Japan|      cornell.edu|       374|
|185.220.128.248|       Belgium|       weebly.com|       389|
|   151.77.19.45|   Afghanistan|independent.co.uk|       282|
|  9.161

In [10]:
#Add to Column to say yes or no whether the country is mexicow
from pyspark.sql.functions import when, col
df_for_new_col = dataframe.withColumn(
                "IS_MEXICO",
                 when(dataframe.Country == "Mexico" , 'yes').otherwise('no')
                )
df_for_new_col.show()

+---------------+--------------+-----------------+----------+---------+
|     ip_address|       Country|      Domain Name|Bytes_used|IS_MEXICO|
+---------------+--------------+-----------------+----------+---------+
|  52.81.192.172|         China| odnoklassniki.ru|       463|       no|
| 119.239.207.13|         China|         youtu.be|        51|       no|
|  68.69.217.210|         China|        adobe.com|        10|       no|
|   7.191.21.223|      Bulgaria|     linkedin.com|       853|       no|
|   211.13.10.68|     Indonesia|          hud.gov|        29|       no|
|   239.80.21.97|      Suriname|       smh.com.au|       218|       no|
|106.214.106.233|       Jamaica|    amazonaws.com|        95|       no|
| 127.242.24.138|         China| surveymonkey.com|       123|       no|
|     99.2.6.139|Czech Republic|     geocities.jp|       322|       no|
|   237.54.11.63|         China|       amazon.com|        83|       no|
| 252.141.157.25|         Japan|      cornell.edu|       374|   

In [24]:
from pyspark.sql.functions import *

df_group = df_for_new_col.groupBy("IS_MEXICO").agg(sum(col("Bytes_used")).alias("n"))

df_group.show()

+---------+--------+
|IS_MEXICO|       n|
+---------+--------+
|       no|508076.0|
|      yes|  6293.0|
+---------+--------+



In [28]:
# Group by country & use sqlfunc.countDistinct calcuate the number of IB
import pyspark.sql.functions as sqlfunc
df = df_for_new_col.groupBy("Country").agg(sqlfunc.countDistinct("ip_address").alias("new"))
df.sort(col("new").desc()).show()

+--------------+---+
|       Country|new|
+--------------+---+
|         China|172|
|     Indonesia|114|
|   Philippines| 65|
|        Russia| 56|
|        Brazil| 35|
|        Poland| 31|
|        Sweden| 28|
|         Japan| 25|
|Czech Republic| 23|
|      Portugal| 23|
|        France| 21|
|          Peru| 19|
|      Colombia| 17|
| United States| 15|
|       Ukraine| 14|
|     Argentina| 14|
|        Mexico| 13|
|      Thailand| 12|
|       Nigeria| 11|
|        Canada| 11|
+--------------+---+
only showing top 20 rows

